# Generative Adversarial Networks

Generative adversarial networks offer a radically different answer to the generative modeling problem. Instead of writing a tractable density and optimizing its log-likelihood or a lower bound, GANs learn through competition. A generator network $G_\theta$ transforms a simple latent variable $\boldsymbol{z} \sim p(\boldsymbol{z})$ into a synthetic image $G_\theta(\boldsymbol{z})$. A discriminator network $D_\psi$ receives either a real image or a generated one and outputs a score interpreted as the probability that the input came from the data distribution. Training is then expressed as a two-player game in which the discriminator tries to distinguish real from fake, while the generator tries to fool the discriminator.

This formulation is historically important because it demonstrated that one can learn a high-quality implicit generative model without evaluating a density explicitly. The induced model distribution is the pushforward of the latent prior through the generator. Sampling is easy. Density evaluation is generally intractable. The training signal comes from the discriminator, which adapts during learning and effectively provides a learned loss.

As with the VAE chapter, the goal here is not to separate theory and code into unrelated reading experiences. GANs are especially sensitive to architecture and optimization details, so the implementation discussion belongs inside the conceptual chapter. In adversarial learning, tiny practical choices often change the training dynamics dramatically.

The chapter is placed immediately after VAEs because the contrast is pedagogically sharp. VAEs showed how far one can go by keeping a probabilistic model explicit and paying the price of a variational approximation. GANs ask the opposite question: what happens if we give up tractable likelihoods altogether and instead learn by comparison between real and generated samples. Reading the two chapters back to back is one of the best ways to see that deep generative modeling is not defined by one canonical objective, but by a family of compromises between tractability, sample quality, interpretability, and optimization behavior.

There is also a pedagogical reason GANs deserve special care in a PhD course. The basic equations are short enough that students may believe the method is conceptually simpler than VAEs or diffusion models. In one sense that is true: the formalism is compact. In another sense it is misleading, because the compactness hides a difficult optimization problem. GANs teach an important lesson that recurs throughout modern machine learning: a model can be easy to state, elegant in its idealized theory, and still delicate in numerical practice. For this reason, the GAN chapter should be read simultaneously as a study of implicit generative modeling and as a study of what happens when optimization itself becomes part of the model's behavior.

The contrast with likelihood-based models is especially valuable. In a VAE we had a clearly identified probability model, a lower bound, and an explicit approximate posterior. In a GAN we no longer ask the model to assign a tractable density to an image. Instead, we ask whether generated samples can be distinguished from real ones by a learned critic. The training objective is therefore relational rather than absolute. The generator is judged not by a fixed score function but by the current state of its adversary. This makes the method powerful, but it also means that training dynamics matter as much as the static objective.

## The Minimax Objective

Let $p_{gt}$ be the data distribution on images and let $p_\theta$ be the distribution induced by $G_\theta(\boldsymbol{z})$ when $\boldsymbol{z} \sim p(\boldsymbol{z})$. The original GAN objective is
:::{math}
\min_\theta \max_\psi V(D_\psi, G_\theta)
=
\mathbb{E}_{\boldsymbol{x} \sim p_{gt}}[\log D_\psi(\boldsymbol{x})]
+
\mathbb{E}_{\boldsymbol{z} \sim p(\boldsymbol{z})}
[\log(1 - D_\psi(G_\theta(\boldsymbol{z})))].
:::
It is useful to interpret the two terms separately. The first rewards the discriminator for assigning large probability to real images. The second rewards it for assigning small probability to generated images. The generator appears only in the second term, where it tries to make synthetic samples difficult to reject.

From a non-technical perspective, one may think of the discriminator as a continuously improving critic and the generator as a continuously improving counterfeiter. The success of the method comes from the fact that the critic is not fixed. It co-evolves with the generator and therefore provides a task-adaptive loss that is much more informative than a simple pixelwise comparison between a generated image and a single target image.

The objective also deserves interpretation as a classification problem. If real and generated samples are mixed with equal prior probability, then the discriminator is solving a binary decision task whose positive class is "came from the data" and whose negative class is "came from the generator." The log terms in the GAN objective are therefore exactly the log-likelihood terms of a probabilistic classifier. This observation is not merely cosmetic. It explains why the discriminator learns quickly at the beginning of training: early fake samples are often easy to reject, so the classification task is simple. The difficulty begins when one asks whether the gradients produced by that classifier remain useful for improving the generator.

At a deeper level, GANs replace direct probabilistic modeling of $p_{gt}$ by a test of indistinguishability. The generator succeeds when generated samples become hard to separate from real ones by a critic drawn from a rich function class. This connects GANs to a broader family of ideas in statistics and machine learning where distributions are compared through witnesses, critics, or integral probability metrics rather than through tractable density formulas.

```{prf:theorem} Optimal discriminator for a fixed generator
:label: thm-gan-optimal-discriminator

Fix a generator and its induced distribution $p_\theta$. For every point $\boldsymbol{x}$ such that $p_{gt}(\boldsymbol{x}) + p_\theta(\boldsymbol{x}) > 0$, the discriminator that maximizes the GAN value function is
:::{math}
D^\star(\boldsymbol{x})
=
\frac{p_{gt}(\boldsymbol{x})}{p_{gt}(\boldsymbol{x}) + p_\theta(\boldsymbol{x})}.
:::
```

```{prf:proof}
For a fixed generator, the objective can be written pointwise as
:::{math}
V(D, G_\theta)
=
\int
p_{gt}(\boldsymbol{x}) \log D(\boldsymbol{x})
+
p_\theta(\boldsymbol{x}) \log(1 - D(\boldsymbol{x}))
\, d\boldsymbol{x}.
:::
Since the integrand decouples in $\boldsymbol{x}$, we maximize
:::{math}
f(d) = a \log d + b \log(1-d)
:::
for $a = p_{gt}(\boldsymbol{x})$ and $b = p_\theta(\boldsymbol{x})$. Differentiating gives
:::{math}
f'(d) = \frac{a}{d} - \frac{b}{1-d}.
:::
Setting this derivative equal to zero yields
:::{math}
a(1-d) = bd,
\qquad
d = \frac{a}{a+b}.
:::
Because
:::{math}
f''(d) = -\frac{a}{d^2} - \frac{b}{(1-d)^2} < 0,
:::
this critical point is the unique maximizer. Substituting back $a = p_{gt}(\boldsymbol{x})$ and $b = p_\theta(\boldsymbol{x})$ gives the claimed formula.
```

The formula for $D^\star$ is revealing. An optimal discriminator does not memorize examples individually. It compares the relative mass that the real and generated distributions assign to each region of image space. Regions where the generator oversamples relative to the data are penalized, and regions where the data dominate are rewarded. This is the first hint that adversarial learning is implicitly minimizing a divergence between distributions rather than matching samples one by one.

It is worth pausing here because this theorem often changes how students think about GANs. The discriminator is sometimes described informally as "detecting realism," but the formula is more precise. It says that the ideal discriminator estimates a density ratio. In regions where the generator places too much mass relative to the data, the discriminator moves downward. In regions where the data dominate, it moves upward. The generator therefore receives a signal about where its distribution is misallocated, even though no explicit density is ever evaluated. This is one of the most conceptually beautiful aspects of the GAN framework.

```{prf:theorem} The GAN game and Jensen-Shannon divergence
:label: thm-gan-jsd

Let $D^\star$ be the optimal discriminator for a fixed generator. Then
:::{math}
V(D^\star, G_\theta)
=
-\log 4 + 2 \operatorname{JSD}(p_{gt} \| p_\theta),
:::
where $\operatorname{JSD}$ denotes the Jensen-Shannon divergence. Consequently, if the discriminator is optimal at each step, minimizing the GAN objective with respect to the generator amounts to minimizing the Jensen-Shannon divergence between the data and model distributions.
```

```{prf:proof}
Substitute the optimal discriminator into the value function:
:::{math}
V(D^\star, G_\theta)
=
\int p_{gt}(\boldsymbol{x})
\log \frac{p_{gt}(\boldsymbol{x})}{p_{gt}(\boldsymbol{x}) + p_\theta(\boldsymbol{x})}
\, d\boldsymbol{x}
+
\int p_\theta(\boldsymbol{x})
\log \frac{p_\theta(\boldsymbol{x})}{p_{gt}(\boldsymbol{x}) + p_\theta(\boldsymbol{x})}
\, d\boldsymbol{x}.
:::
Let
:::{math}
m(\boldsymbol{x}) = \frac{1}{2}(p_{gt}(\boldsymbol{x}) + p_\theta(\boldsymbol{x})).
:::
Then
:::{math}
\log \frac{p_{gt}(\boldsymbol{x})}{p_{gt}(\boldsymbol{x}) + p_\theta(\boldsymbol{x})}
=
\log \frac{p_{gt}(\boldsymbol{x})}{2m(\boldsymbol{x})}
=
\log \frac{p_{gt}(\boldsymbol{x})}{m(\boldsymbol{x})} - \log 2,
:::
and similarly for the second term. Therefore
:::{math}
V(D^\star, G_\theta)
=
\operatorname{KL}(p_{gt} \| m)
+
\operatorname{KL}(p_\theta \| m)
-
2 \log 2.
:::
By definition,
:::{math}
\operatorname{JSD}(p_{gt} \| p_\theta)
=
\frac{1}{2}\operatorname{KL}(p_{gt} \| m)
+
\frac{1}{2}\operatorname{KL}(p_\theta \| m).
:::
Hence
:::{math}
V(D^\star, G_\theta)
=
2 \operatorname{JSD}(p_{gt} \| p_\theta) - \log 4.
:::
This proves the claim.
```

The theorem is elegant, but it must be interpreted carefully. In practice the discriminator is never optimized exactly, the generator and discriminator are finite neural networks, and stochastic gradient descent only approximates the game dynamics. The Jensen-Shannon interpretation is therefore a guiding idealization rather than a literal description of every training step. Still, it explains why GANs are not arbitrary heuristics. They instantiate a principled distribution-matching game.

This is also the right place to explain why the theory can look stronger than the practice. The Jensen-Shannon divergence is perfectly meaningful when both distributions are fixed objects and the discriminator is optimized over all measurable functions. But actual GAN training lives in a restricted and moving landscape: the discriminator is a finite neural network, the generator changes after every update, minibatches add stochastic noise, and the two players may learn on different time scales. As a consequence, one should think of the theorem as describing the ideal geometry that motivates the method, not as proving that every practical run of SGD is faithfully minimizing Jensen-Shannon divergence step by step.

A further subtlety becomes important in high-dimensional image generation. When the supports of $p_{gt}$ and $p_\theta$ are far apart or lie on thin lower-dimensional sets, the Jensen-Shannon divergence can become locally uninformative. Intuitively, if the discriminator can separate real and fake samples almost perfectly, then its output may saturate near zero or one on most of the relevant regions, and the generator may receive poor directional guidance. This is one of the reasons the original GAN formulation produced stunning visual results but also inspired a large stabilization literature. The issue is not that the theory is wrong. The issue is that the geometry induced by Jensen-Shannon divergence may be awkward for gradient-based optimization when distributions overlap weakly.

## Why the Original Generator Loss Can Saturate

If we follow the minimax formulation literally, the generator minimizes
:::{math}
\mathbb{E}_{\boldsymbol{z} \sim p(\boldsymbol{z})}
[\log(1 - D_\psi(G_\theta(\boldsymbol{z})))].
:::
Early in training, however, the discriminator often becomes very good before the generator has learned anything meaningful. In that regime, $D_\psi(G_\theta(\boldsymbol{z}))$ may be close to zero, and the gradient received by the generator can become weak or poorly conditioned. For this reason, practical GAN training often replaces the minimax generator loss with the non-saturating alternative
:::{math}
\mathcal{L}_{G,\mathrm{NS}}(\theta)
=
- \mathbb{E}_{\boldsymbol{z} \sim p(\boldsymbol{z})}
[\log D_\psi(G_\theta(\boldsymbol{z}))].
:::
This alternative has the same fixed point, in the sense that it still encourages generated samples to move toward regions that the discriminator classifies as real, but it tends to provide stronger gradients when the generator is weak.

The word *saturate* deserves a more explicit explanation than it often receives in brief presentations. Suppose the generator is poor and the discriminator is already very confident, so that $D_\psi(G_\theta(\boldsymbol{z})) \approx 0$ for most latent samples. In the minimax loss, the quantity $\log(1 - D_\psi(G_\theta(\boldsymbol{z})))$ is then close to $\log 1 = 0$, and the derivative that reaches the generator may become very small after passing through the discriminator's nonlinearities. The problem is not simply that the loss value is large or small. The problem is that the gradient field can become weak exactly when the generator most needs useful corrective information.

The non-saturating loss changes the emphasis. Instead of minimizing $\log(1 - D_\psi(G_\theta(\boldsymbol{z})))$, it maximizes $\log D_\psi(G_\theta(\boldsymbol{z}))$ or equivalently minimizes its negative. When the discriminator is confident that fake samples are fake, this objective penalizes the generator more aggressively and often produces stronger gradients. The fixed point is unchanged in the idealized setting because both objectives encourage fake samples to move toward regions judged real. But the local training dynamics can be much better. This is a central example of a broader principle in deep learning: mathematically equivalent goals at equilibrium can behave very differently under gradient descent.

One can push this lesson further. GAN research repeatedly shows that choosing a useful optimization surrogate matters as much as choosing the final objective being approximated. Two players may share the same desired equilibrium while differing substantially in whether their gradients guide them there efficiently. This perspective helps students connect GANs to later topics such as score matching and flow matching, where the practical success of the method also depends on rewriting a difficult objective into a regression problem with better gradients.

## Mode Collapse

One of the central pathologies of GAN training is mode collapse. The data distribution may have many distinct modes corresponding, for instance, to different object classes, poses, or textures. A generator trained adversarially can sometimes discover that producing only a small subset of visually convincing outputs is sufficient to fool the current discriminator. As a result, sample quality may appear high while sample diversity is poor.

This phenomenon reflects the fact that the adversarial game is a dynamic optimization problem rather than a static convex program. The discriminator reacts to the generator, the generator reacts to the discriminator, and the resulting vector field in parameter space can exhibit oscillations, local instabilities, and partial equilibria that do not represent the full data distribution well. In lectures, this is an excellent place to emphasize that a powerful objective on paper does not automatically imply easy optimization in practice.

A helpful intuition is to imagine a dataset containing several well-separated semantic groups, such as shoes, bags, and coats. If the current discriminator is not yet sensitive to the absence of some groups, the generator may learn that producing only one or two convincing categories yields a strong short-term reward. Once that happens, the discriminator eventually adapts, but the generator may then jump toward a different narrow subset rather than spreading its mass more evenly across the full data distribution. The resulting training trajectory can oscillate between partial solutions instead of converging to broad coverage.

This interpretation is useful because it prevents a common misunderstanding. Mode collapse is not simply "the generator repeats the same picture" in a trivial memorization sense. It is a distributional failure in which the generator allocates probability mass too narrowly relative to the diversity of the data. The samples may still look sharp and varied at first glance, especially to a casual observer. The failure becomes visible only when one asks whether the whole dataset is represented fairly. This is one reason evaluation in GANs is subtle: visual inspection alone is informative but not sufficient.

Several remedies can be interpreted as attempts to give the discriminator a smoother or more globally meaningful signal. Feature matching encourages the generator to match intermediate discriminator statistics rather than only the final decision boundary. Minibatch discrimination lets the discriminator look at relationships among samples in a batch so it can detect suspicious lack of diversity. Wasserstein objectives change the underlying geometry so that moving mass between modes produces a more graded penalty. None of these ideas completely solves adversarial optimization, but together they illustrate the central point that diversity failures are tied to the structure of the game, not merely to insufficient model capacity.

## Variants and Stabilization Strategies

A large part of the GAN literature can be read as an attempt to stabilize the adversarial game or to replace the underlying divergence with a better behaved discrepancy. Wasserstein GANs replace the Jensen-Shannon geometry with the Wasserstein-1 distance, leading to smoother gradients when the supports of the data and generator distributions barely overlap. Spectral normalization constrains the Lipschitz constant of the discriminator layer by layer and often improves stability dramatically with very little implementation overhead. CycleGAN extends the adversarial principle to unpaired image-to-image translation by combining adversarial losses with a cycle-consistency penalty that encourages the forward and backward mappings to preserve content.

```{figure} ../assets/images/GAN_architecture.png
:width: 76%
:align: center

A standard GAN architecture with a generator and a discriminator trained in opposition.
```

Wasserstein GAN is especially important because it changes not only a training heuristic but the geometry of the problem. The Wasserstein-1 distance measures the cost of transporting probability mass from the model distribution to the data distribution. Unlike Jensen-Shannon divergence, it can vary continuously even when the distributions have nearly disjoint support. This is exactly the regime one often faces early in GAN training. The Kantorovich-Rubinstein duality then rewrites the transport distance as a supremum over 1-Lipschitz critics, which explains why Lipschitz control becomes central in WGAN training.

The original WGAN paper enforced the Lipschitz constraint by weight clipping, which made the idea historically influential but also introduced optimization pathologies of its own. Later work used gradient penalties or spectral normalization to impose smoother constraints. From a teaching perspective, this is a beautiful example of an abstract mathematical condition becoming a concrete implementation problem. One first proves that a certain function class gives the right dual formulation, and then one must still decide how a neural network should be constrained to live approximately inside that class.

Spectral normalization deserves separate emphasis because it is one of the simplest practically successful stabilizers. If each linear map in the discriminator is divided by an estimate of its largest singular value, then the operator norm of that layer is controlled. While the global Lipschitz constant of a deep network is more subtle than a product of per-layer constants, this procedure often gives a strong enough approximation to improve stability substantially. Students sometimes appreciate spectral normalization because it shows that not every important GAN idea requires rewriting the whole objective; sometimes a carefully chosen architectural constraint changes the training dynamics dramatically.

CycleGAN illustrates a different axis of variation. The adversarial mechanism is not limited to unconditional generation from noise. It can be embedded in structured tasks such as image-to-image translation. When paired data are unavailable, adversarial losses alone do not determine a unique semantic mapping between domains. The cycle-consistency term addresses this by requiring that a sample translated from domain A to domain B and back again should recover the original content approximately. The resulting model is a reminder that GANs are best understood as a training principle that can be combined with additional inductive biases and reconstruction terms, not only as one standalone architecture.

The conceptual position of GANs within the course is now clear. VAEs emphasized explicit probabilistic structure and a tractable lower bound. GANs sacrifice explicit likelihood evaluation and instead learn through an adaptive critic. Diffusion models will later revisit explicit probabilistic reasoning, but through denoising trajectories rather than adversarial games. The original GAN formulation is due to {cite}`goodfellow2014generative`, while influential stabilization directions include {cite}`arjovsky2017wasserstein`, {cite}`miyato2018spectral`, and {cite}`zhu2017unpaired`.

## A Guided Vanilla GAN Implementation

The minimal GAN code is not long, but it repays careful reading. The generator and discriminator are simple enough to fit in a lecture notebook, yet they already expose the central adversarial mechanics: data scaling, generator upsampling, discriminator downsampling, non-saturating generator loss, and alternating optimization.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
if device.type == "cuda":
    torch.cuda.manual_seed_all(7)
num_workers = 2 if device.type == "cuda" else 0
project_root = Path.cwd() if (Path.cwd() / "_config.yml").exists() else Path.cwd().parent
DATA_ROOT = project_root / "data"

# DCGAN-style settings are still teachable but produce much better samples.
latent_dim = 128
base_channels = 64
batch_size = 128
lr = 2e-4
epochs = 60

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 2.0 * x - 1.0),
])

train_dataset = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)


The hyperparameters here are no longer set as if this were just a ten-minute smoke test. `FashionMNIST` is still a small benchmark, but GANs usually need a longer optimization horizon than one might guess from compact tutorials, especially if the goal is not merely to see the losses move but to obtain samples that actually look coherent.

For that reason, the vanilla GAN in this chapter is trained for `60` epochs with a DCGAN-style architecture and standard Adam settings. That is still far from an industrial training run, but it is long enough that the model has a fair chance to develop recognizable structure instead of staying in the regime of obviously immature generations.


## Generator and Discriminator Architectures

```{figure} ../assets/images/GAN_architecture.png
:width: 76%
:align: center

A standard GAN architecture: a generator maps noise to images, while a discriminator learns to distinguish real samples from generated ones.
```

The generator starts from a latent vector and progressively expands it into spatial feature maps. The discriminator performs the opposite movement, reducing an image to a realism score. This asymmetry is not accidental. One network is learning how to *synthesize* spatial structure, the other how to *detect* whether that structure looks data-like.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=128, base_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, base_channels * 4 * 7 * 7),
            nn.BatchNorm1d(base_channels * 4 * 7 * 7),
            nn.ReLU(True),
            nn.Unflatten(1, (base_channels * 4, 7, 7)),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(True),
            nn.Conv2d(base_channels, 1, kernel_size=3, padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(base_channels * 2 * 7 * 7, 1),
        )

    def forward(self, x):
        return self.net(x)


G = Generator(latent_dim=latent_dim, base_channels=base_channels).to(device)
D = Discriminator(base_channels=base_channels).to(device)

g_optimizer = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

Compared with the VAE, the architecture already reflects a different philosophy. There is no explicit encoder and no tractable likelihood. The discriminator is not a probabilistic posterior approximation. It is an adaptive critic whose only job is to shape the generator's gradients. That difference is why GAN code can be shorter than VAE code while still being harder to tune.

In [ ]:
def discriminator_loss(real_logits, fake_logits):
    # Mild one-sided label smoothing usually stabilizes the discriminator.
    real_targets = torch.full_like(real_logits, 0.9)
    fake_targets = torch.zeros_like(fake_logits)

    real_loss = F.binary_cross_entropy_with_logits(real_logits, real_targets)
    fake_loss = F.binary_cross_entropy_with_logits(fake_logits, fake_targets)
    return real_loss + fake_loss


def generator_loss(fake_logits):
    # Non-saturating loss: the generator wants fake samples judged as real.
    real_targets = torch.ones_like(fake_logits)
    return F.binary_cross_entropy_with_logits(fake_logits, real_targets)

This pair of functions maps directly onto the theory. The discriminator loss combines one term for real images and one for fake images. The generator loss is the non-saturating alternative, so fake logits are compared against a target of one. In words, the generator is trained as if its own samples ought to be classified as real. Students often find this clearer when said explicitly: the generator never sees real images directly in its loss. It only sees how the discriminator currently reacts to its own outputs.

If one wanted to implement the literal minimax generator objective instead, the code would be very short to change. The theory notebook explains why we usually do not make that change in practice. This is an excellent point to encourage students to modify the loss and compare the training behavior themselves later, even if the main course notes avoid turning every notebook into an exercise sheet.

## Alternating Optimization in Practice

The training loop is where GANs stop looking like ordinary supervised models. We first update the discriminator on real data and on **detached** fake samples. Then we update the generator against the current discriminator. The `detach()` call is not cosmetic: it prevents the generator from receiving gradients during the discriminator step.

In [ ]:
history = {"d_loss": [], "g_loss": []}


fixed_z = torch.randn(16, latent_dim, device=device)

for epoch in tqdm(range(epochs), desc="GAN epochs"):
    d_running = 0.0
    g_running = 0.0

    for real_images, _ in tqdm(train_loader, desc="train", leave=False):
        real_images = real_images.to(device)
        batch_n = real_images.size(0)

        # First update the discriminator on real and detached fake samples.
        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = G(z)

        d_optimizer.zero_grad()
        real_logits = D(real_images)
        fake_logits = D(fake_images.detach())
        d_loss = discriminator_loss(real_logits, fake_logits)
        d_loss.backward()
        d_optimizer.step()

        # Then update the generator against the current discriminator.
        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = G(z)

        g_optimizer.zero_grad()
        fake_logits = D(fake_images)
        g_loss = generator_loss(fake_logits)
        g_loss.backward()
        g_optimizer.step()

        d_running += d_loss.item()
        g_running += g_loss.item()

    d_epoch = d_running / len(train_loader)
    g_epoch = g_running / len(train_loader)
    history["d_loss"].append(d_epoch)
    history["g_loss"].append(g_epoch)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"D loss: {d_epoch:.4f} | "
        f"G loss: {g_epoch:.4f}"
    )

This alternating structure is the main reason GAN training feels dynamic rather than static. The loss landscape is being reshaped after every step because one player changes the objective seen by the other. This is why interpreting GAN losses requires more care than interpreting VAE losses. A rising generator loss is not automatically bad, and a tiny discriminator loss is not automatically good.

In [ ]:
@torch.no_grad()
def show_gan_samples(generator, device, n=16):
    generator.eval()
    z = fixed_z[:n] if n <= fixed_z.size(0) else torch.randn(n, latent_dim, device=device)
    samples = generator(z).view(-1, 1, 28, 28)
    # Undo tanh scaling for display.
    samples = 0.5 * (samples + 1.0)
    image = utils.make_grid(samples.cpu(), nrow=4, pad_value=1.0)
    plt.figure(figsize=(6, 6))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()
    generator.train()


show_gan_samples(G, device)

When this notebook works reasonably well, one should expect the generator to move from amorphous noise toward recognizable clothing silhouettes, often with sharper edges than a comparably small VAE produces. At the same time, one should also expect imperfections: repeated shapes, missing classes, unstable textures, or sensitivity to random initialization. Those imperfections are not incidental. They are part of what the notebook is meant to teach.

If training fails badly, a few debugging clues are especially common. If all samples look nearly identical, suspect mode collapse or an overpowered discriminator. If both losses oscillate wildly and samples never improve, learning rates may be too large or the network balance may be poor. If samples remain pure noise while the discriminator loss drops immediately, the discriminator may be learning too fast relative to the generator. These are not exhaustive diagnoses, but they help students see adversarial training as something one interprets dynamically rather than passively runs.

## Distribution-Level Evaluation

Because GANs can look sharp while still collapsing onto too few modes, quantitative distribution-level metrics are particularly valuable here. FID and KID do not solve evaluation completely, but they provide a useful complement to image grids and make the discussion of mode collapse more concrete.

In [ ]:
def prepare_for_inception_metrics(images):
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    return images.clamp(0.0, 1.0)


@torch.no_grad()
def compute_gan_fid_and_kid(generator, real_loader, device, num_fake=1000, metric_batch_size=64):
    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True,
        reset_real_features=False,
    ).set_dtype(torch.float64).to(device)
    kid = KernelInceptionDistance(
        feature=2048,
        subsets=10,
        subset_size=100,
        normalize=True,
        reset_real_features=False,
    ).to(device)

    seen_real = 0
    real_target = num_fake
    for real_images, _ in tqdm(real_loader, desc="GAN real metrics", leave=False):
        remaining = real_target - seen_real
        if remaining <= 0:
            break
        real_images = real_images[: min(metric_batch_size, remaining)].to(device)
        real_images = prepare_for_inception_metrics(0.5 * (real_images + 1.0))
        fid.update(real_images, real=True)
        kid.update(real_images, real=True)
        seen_real += real_images.size(0)

    generated = 0
    pbar = tqdm(total=num_fake, desc="GAN fake metrics", leave=False)
    while generated < num_fake:
        batch_n = min(metric_batch_size, num_fake - generated)
        z = torch.randn(batch_n, latent_dim, device=device)
        fake_images = generator(z).view(-1, 1, 28, 28)
        # Undo tanh scaling so the metric sees images in [0, 1].
        fake_images = 0.5 * (fake_images + 1.0)
        fake_images = prepare_for_inception_metrics(fake_images)
        fid.update(fake_images, real=False)
        kid.update(fake_images, real=False)
        generated += batch_n
        pbar.update(batch_n)
    pbar.close()

    kid_mean, kid_std = kid.compute()
    return {
        "fid": fid.compute().item(),
        "kid_mean": kid_mean.item(),
        "kid_std": kid_std.item(),
    }


metric_scores = compute_gan_fid_and_kid(G, train_loader, device)
print(metric_scores)

This metric block is especially useful for discussing **mode collapse**. A GAN may improve visually while still obtaining disappointing FID or KID if it allocates mass too narrowly over the dataset. That makes these scores a nice complement to image grids: the image grid shows local sharpness, while FID and KID say something more about global coverage.

The harmonic reading of the chapter should now be clear. The theory explained why the adversarial game is meaningful, why Jensen-Shannon geometry can be problematic, and why saturation or collapse can happen. The implementation showed how those ideas appear as architectural and optimization decisions. That unity matters more in GANs than in almost any other chapter of the course.